## Downaload a dataset

In some cases, you need to login to be able to download a dataset. Run this cell, if that is the case.

In [1]:
from huggingface_hub import login
login()

ModuleNotFoundError: No module named 'huggingface_hub'

On Huggingface, click on the copy button to get the name of the dataset.

![dataset_from_hugging_face](../images/get_dataset_from_hugging_face.png)

And provide that name to `load_dataset`

In [ ]:
from datasets import load_dataset

# Load a specific language subset (recommended — full dataset is huge)
ds = load_dataset("HuggingFaceFW/fineweb-2", name="ita_Latn", split="train", streaming=True)

### Extract and save dataset

By Chinchilla's law (20 tokens/param), a ~86M params model needs ~1.72B tokens. At ~4.5 characters per token, that's ~7.74 GB of raw text — I collect 8 GB to leave a small buffer for filtering losses during tokenization.

In [ ]:
TARGET_GB      = 8                        # stop after this many GB of text
MIN_CHARS      = 100                      # skip very short documents
MAX_CHARS      = 10_000                   # skip absurdly long ones
OUTPUT_FILE    = "../data/fineweb2_raw.txt"

TARGET_BYTES   = TARGET_GB * 1_000_000_000

In [ ]:
total_bytes   = 0
total_docs    = 0
skipped_docs  = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for sample in ds:
        text = sample["text"].replace('\n', ' ')
        length = len(text)

        # Filter by text length
        if length < MIN_CHARS or length > MAX_CHARS:
            skipped_docs += 1
            continue

        f.write(text + '\n')

        total_bytes += len(text.encode("utf-8"))
        total_docs  += 1

        # Progress every 50k docs
        if total_docs % 50_000 == 0:
            print(f"  {total_docs:,} docs | {total_bytes / 1e9:.2f} GB collected")

        if total_bytes >= TARGET_BYTES:
            break

print(f"\nDone!")
print(f"  Documents : {total_docs:,}")
print(f"  Skipped   : {skipped_docs:,}")
print(f"  Size      : {total_bytes / 1e9:.2f} GB")